In [1]:
import os

# Folders to exclude
EXCLUDED_FOLDERS = {'myenv', 'scipy', '__pycache__', '.git','skin-disease-datasaet'}

def print_tree(start_path, indent=""):
    for item in sorted(os.listdir(start_path)):
        item_path = os.path.join(start_path, item)
        if item in EXCLUDED_FOLDERS:
            continue
        if os.path.isdir(item_path):
            print(f"{indent}📁 {item}")
            print_tree(item_path, indent + "    ")
        else:
            print(f"{indent}📄 {item}")

# Replace '.' with your desired root directory
print_tree('.')


📄 llm.ipynb
📄 skin_disease_prediction.ipynb
📄 web_scrapping.ipynb


In [2]:
import pandas as pd
import spacy
import sys
import os

In [3]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from backend.config import MAYO_CSV

In [4]:
df = pd.read_csv(MAYO_CSV)

In [5]:
df.head()

,disease,main_link,Diagnosis_treatment_link,Doctors_departments_link,Overview,Symptoms,When to see a doctor,Causes,Risk factors,Complications,Prevention,Diagnosis,Treatment,Coping and support,Preparing for your appointment,Lifestyle and home remedies
0,Atrial fibrillation,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Atrial fibrillation (AFib) is an irregular and...,Symptoms ofAFibmay include:\nFeelings of a fas...,"If you have symptoms of atrial fibrillation, m...",To understand the causes of atrial fibrillatio...,Things that can increase the risk of atrial fi...,Blood clots are a dangerous complication of at...,Healthy lifestyle choices can reduce the risk ...,You may not know you have atrial fibrillation ...,The goals of atrial fibrillation treatment are...,NaN,If you have an irregular or pounding heartbeat...,Following a heart-healthy lifestyle can help p...
1,Hyperhidrosis,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Hyperhidrosis (hi-pur-hi-DROE-sis) is excessiv...,The main symptom of hyperhidrosis is heavy swe...,Sometimes excessive sweating is a sign of a se...,Sweating is the body's mechanism to cool itsel...,Risk factors for hyperhidrosis include:\nHavin...,Complications of hyperhidrosis include:\nInfec...,NaN,Diagnosing hyperhidrosis may start with your h...,Treating hyperhidrosis may start with treating...,Hyperhidrosis can be the cause of discomfort a...,You may start by seeing your primary care prov...,The following suggestions may help control swe...
2,Bartholin's cyst,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,The Bartholin's (BAHR-toe-linz) glands are loc...,"If you have a small, noninfected Bartholin's c...",Call your doctor if you have a painful lump ne...,Experts believe that the cause of a Bartholin'...,NaN,A Bartholin's cyst or abscess may recur and ag...,There's no way to prevent a Bartholin's cyst. ...,"To diagnose a Bartholin's cyst, your doctor ma...",Often a Bartholin's cyst requires no treatment...,NaN,Your first appointment will likely be with eit...,NaN
3,Infant reflux,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,NaN,Infant reflux is when a baby spits up liquid o...,"Most of the time, infant reflux isn't a cause ...",See a healthcare professional if a baby:\nIsn'...,"In infants, the ring of muscle between the eso...",Infant reflux is common. But some things make ...,Infant reflux usually gets better on its own. ...,NaN,"To diagnose infant reflux, a healthcare profes...","For most babies, making some changes to feedin...",NaN,You may start by seeing your baby's primary he...,To minimize reflux:\nFeed your baby in an upri...
4,Hidradenitis suppurativa,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Hidradenitis suppurativa (hi-drad-uh-NIE-tis s...,Hidradenitis suppurativa can affect one or sev...,Early diagnosis of hidradenitis suppurativa is...,Hidradenitis suppurativa develops when hair fo...,Factors that increase your chance of developin...,Persistent and severe hidradenitis suppurativa...,NaN,Hidradenitis suppurativa can be mistaken for p...,"Treatment with medicines, surgery or both can ...",Hidradenitis suppurativa can be a challenge to...,You'll likely first see your primary care prov...,Mild hidradenitis suppurativa can sometimes be...


In [6]:
symptoms1 = df["Symptoms"][0]
symptoms2 = df["Symptoms"][1]
symptoms1 , "......................................." ,symptoms2

("Symptoms ofAFibmay include:\nFeelings of a fast, fluttering or pounding heartbeat, called palpitations.\nChest pain.\nDizziness.\nFatigue.\nLightheadedness.\nReduced ability to exercise.\nShortness of breath.\nWeakness.\nSome people with atrial fibrillation (AFib) don't notice any symptoms.\nAtrial fibrillation may be:\nOccasional, also called paroxysmal atrial fibrillation.AFibsymptoms come and go. The symptoms usually last for a few minutes to hours. Some people have symptoms for as long as a week. The episodes can happen repeatedly. Symptoms might go away on their own. Some people with occasionalAFibneed treatment.\nPersistent.The irregular heartbeat is constant. The heart rhythm does not reset on its own. If symptoms occur, medical treatment is needed to correct the heart rhythm.\nLong-standing persistent.This type ofAFibis constant and lasts longer than 12 months. Medicines or a procedure are needed to correct the irregular heartbeat.\nPermanent.In this type of atrial fibrillati

In [8]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(symptoms)

for ent in doc.ents:
    print(ent.text, ent.label_)


AFib ORG
a few minutes to hours TIME
as long as a week DATE
12 months DATE


In [9]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_name = "d4data/biomedical-ner-all"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

entities = ner(symptoms)
for e in entities:
    if e['entity_group'] == 'Sign_symptom':
        print(e['word'], e['entity_group'])


Device set to use cpu


pal Sign_symptom
##pit Sign_symptom
pain Sign_symptom
di Sign_symptom
to Sign_symptom
short Sign_symptom
afibsymptoms Sign_symptom
symptoms Sign_symptom
symptoms Sign_symptom
irregular heartbeat Sign_symptom
symptoms Sign_symptom


In [9]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage
import os

# For Groq
os.environ["OPENAI_API_KEY"] = os.getenv("GROQ_API_KEY") or "your_groq_key"
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model="llama3-70b-8192",  # or llama3-70b-8192 if supported
    temperature=0
)



C:\Users\harme\AppData\Local\Temp\ipykernel_12644\3804315171.py:9: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [17]:
from langchain.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables=['symptoms'],
template = """
You are a medical text processor. 
Your task is to carefully read the following paragraph where a patient casually describes their health condition. 

Extract **only the medical symptoms** mentioned. 
- ❌ Do NOT include: disease names, extra text, explanations, or labels. 
- ✅ Output should be: a simple, clean, comma-separated list of symptom words/phrases, all in lowercase. 
- ✅ Remove duplicates and unnecessary words.
- ✅ Output ONLY the symptoms, nothing else.
- No extra words explaination or nothing is required , no need of " ' " this symbols too

Paragraph: {symptoms}

Symptoms (comma-separated):
"""

)


In [18]:
chain = prompt_template_name | llm


In [21]:
response = chain.invoke({"symptoms":symptoms2})

In [25]:
symptom_list = [s.strip() for s in response.content.split(",")]
symptom_list

['heavy sweating', 'anxious', 'stressed']

In [57]:
import spacy

# Load the biomedical NER model
nlp = spacy.load("en_ner_bc5cdr_md")

def extract_entities(text):
    doc = nlp(text)
    symps = list({ent.text.lower() for ent in doc.ents})
    return symps
        


c:\Users\harme\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_ner_bc5cdr_md' (0.5.1) was trained with spaCy v3.4.1 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [41]:
extract_entities(symptoms)

['palpitations',
 'dizziness',
 'chest pain',
 'atrial fibrillation',
 'shortness of breath',
 'pounding',
 'lightheadedness',
 'afib']

In [42]:
from langchain.schema import Document

In [58]:
disease1 = df["disease"][0]
disease2 = df["disease"][1]
disease1, disease2

('Atrial fibrillation', 'Hyperhidrosis')

In [44]:
sympto =    extract_entities(symptoms)
sympto

['palpitations',
 'dizziness',
 'chest pain',
 'atrial fibrillation',
 'shortness of breath',
 'pounding',
 'lightheadedness',
 'afib']

In [47]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [48]:
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\harme\AppData\Local\Temp\ipykernel_3124\18197047.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [66]:
df.shape[0]

1093

In [69]:
for i in range(df.shape[0]):
    print(df["disease"][i])
    break

Atrial fibrillation


In [71]:
df.columns

Index(['disease', 'main_link', 'Diagnosis_treatment_link',
       'Doctors_departments_link', 'Overview', 'Symptoms',
       'When to see a doctor', 'Causes', 'Risk factors', 'Complications',
       'Prevention', 'Diagnosis', 'Treatment', 'Coping and support',
       'Preparing for your appointment', 'Lifestyle and home remedies'],
      dtype='object')

In [ ]:
from tqdm import tqdm

disease_db = {}

for _, row in tqdm(df.iterrows(), total=len(df), desc="🔍 Processing diseases"):
    disease = row["disease"]
    
    # ✅ Handle NaN / missing symptoms
    symptoms_text = row["Symptoms"]
    if not isinstance(symptoms_text, str):
        continue  # Skip rows with no symptoms

    # Extract symptoms
    symptom_list = extract_entities(symptoms_text)

    # If no symptoms extracted, skip
    if not symptom_list:
        response = chain.invoke({"symptoms": symptoms_text})
        symptom_list = [s.strip() for s in response.content.split(",")]

        

    # Generate embeddings
    symptom_embeddings = embedder.embed_documents(symptom_list)

    disease_db[disease] = {
        "symptoms": symptom_list,
        "embeddings": symptom_embeddings
    }

print(f"\n✅ Processed {len(disease_db)} diseases successfully.")


🔍 Processing diseases: 100%|██████████| 1093/1093 [00:43<00:00, 25.07it/s]


✅ Processed 647 diseases successfully.


In [ ]:
# Example structure
disease_db = {
    disease1: {
        "symptoms": symptoms1,
        "embeddings": embedder.embed_documents(extract_entities(symptoms1))
    },
    disease2: {
        "symptoms": symptoms2,
        "embeddings": embedder.embed_documents(extract_entities(symptoms2))
    }
}


In [60]:
disease_db

{'Atrial fibrillation': {'symptoms': "Symptoms ofAFibmay include:\nFeelings of a fast, fluttering or pounding heartbeat, called palpitations.\nChest pain.\nDizziness.\nFatigue.\nLightheadedness.\nReduced ability to exercise.\nShortness of breath.\nWeakness.\nSome people with atrial fibrillation (AFib) don't notice any symptoms.\nAtrial fibrillation may be:\nOccasional, also called paroxysmal atrial fibrillation.AFibsymptoms come and go. The symptoms usually last for a few minutes to hours. Some people have symptoms for as long as a week. The episodes can happen repeatedly. Symptoms might go away on their own. Some people with occasionalAFibneed treatment.\nPersistent.The irregular heartbeat is constant. The heart rhythm does not reset on its own. If symptoms occur, medical treatment is needed to correct the heart rhythm.\nLong-standing persistent.This type ofAFibis constant and lasts longer than 12 months. Medicines or a procedure are needed to correct the irregular heartbeat.\nPermane

In [76]:
from sklearn.metrics.pairwise import cosine_similarity

def match_disease(user_input, disease_db, threshold=0.7):
    query_symptoms = extract_entities(user_input)
    query_embeddings = embedder.embed_documents(query_symptoms)

    results = []

    for disease, data in disease_db.items():
        scores = cosine_similarity(query_embeddings, data["embeddings"])
        max_similarities = scores.max(axis=1)
        coverage = (max_similarities > threshold).mean()
        avg_similarity = max_similarities.mean()

        results.append((disease, coverage, avg_similarity))

    # Rank: first by coverage, then avg similarity
    results = sorted(results, key=lambda x: (-x[1], -x[2]))
    return results[:3]  # top 3 diseases


In [88]:
df["disease"][600]

'Patellar tendinitis'

In [89]:
match_disease(df["Symptoms"][600], disease_db)

[('Patellar tendinitis', 1.0, 1.0000000000000004),
 ('Patellofemoral pain syndrome', 1.0, 0.8826984745963159),
 ('Achilles tendinitis', 0.5, 0.8486648739607416)]